# Load Pre-trained Embedding Model

In [1]:
from sentence_transformers import SentenceTransformer
from datasets import load_dataset, concatenate_datasets
import torch

model_id = "Snowflake/snowflake-arctic-embed-m"  # Use a reasonably good model here

model_retrieval = SentenceTransformer(
    model_id, device="cuda" if torch.cuda.is_available() else "cpu"
)

# Load and Combine Multiple Datasets

In [2]:
from datasets import load_dataset, concatenate_datasets

# Load each split separately
easy_para = load_dataset("dnth/ssf-dataset-synthetic-v2", "easy_triplets")["train"]
hard_para = load_dataset("dnth/ssf-dataset-synthetic-v2", "hard_triplets")["train"]
# hard_sem = load_dataset("frankwong2001/ssf-dataset_Full_synthetic_batch10", "hard_triplets_semantic")["train"]

# Concatenate all splits into one dataset
dataset = concatenate_datasets([easy_para, hard_para,])

README.md: 0.00B [00:00, ?B/s]

easy_triplets/train-00000-of-00001.parqu(…):   0%|          | 0.00/4.74M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1885 [00:00<?, ? examples/s]

hard_triplets/train-00000-of-00001.parqu(…):   0%|          | 0.00/4.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1885 [00:00<?, ? examples/s]

# Clean Dataset Structure

In [3]:
dataset = dataset.select_columns(['anchor', 'positive', 'negative'])
dataset

Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 3770
})

# Define Quality Assessment Functions
### These functions compute semantic similarity scores between different text pairs in our triplets:
- get_embeddings(): Converts text to vector representations using our embedding model
- get_similarities(): Computes cosine similarity between embedding vectors
- format_data_retriever(): Processes batches of triplets to add similarity scores

### The similarity scores help us identify high-quality triplets where:
- Anchor-positive pairs have high similarity (good matches)
- Anchor-negative pairs have moderate similarity (hard negatives)
- Positive-negative pairs are sufficiently different

In [4]:
from sklearn.metrics.pairwise import cosine_similarity

def get_embeddings(texts):
    vectors = model_retrieval.encode(texts)
    return [vector.tolist() for vector in vectors]


def get_similarities(vector_batch_a, vector_batch_b):
    similarities = []
    for vector_a, vector_b in zip(vector_batch_a, vector_batch_b):
        similarity = cosine_similarity([vector_a], [vector_b])[0][0]
        similarities.append(similarity)
    return similarities

def format_data_retriever(batch):# -&gt; Any:
    batch["anchor-vector"] = get_embeddings(batch["anchor"])
    batch["positive-vector"] = get_embeddings(batch["positive"])
    batch["negative-vector"] = get_embeddings(batch["negative"])    
    batch["similarity-positive-negative"] = get_similarities(batch["positive-vector"], batch["negative-vector"])
    batch["similarity-anchor-positive"] = get_similarities(batch["anchor-vector"], batch["positive-vector"])
    batch["similarity-anchor-negative"] = get_similarities(batch["anchor-vector"], batch["negative-vector"])
    return batch

# Compute Similarity Scores

In [5]:
dataset = dataset.map(format_data_retriever, batched=True, batch_size=250)

Map:   0%|          | 0/3770 [00:00<?, ? examples/s]

# Inspect the Enriched Dataset

In [6]:
dataset.to_pandas()

,anchor,positive,negative,anchor-vector,positive-vector,negative-vector,similarity-positive-negative,similarity-anchor-positive,similarity-anchor-negative
0,The Audit Associate/Audit Assistant Associate ...,The Audit Associate/Audit Assistant Associate ...,The Tax Associate specializes in preparing tax...,"[0.06092929095029831, 0.09926585108041763, 0.0...","[0.06152121350169182, 0.08979933708906174, 0.0...","[0.04587974399328232, 0.07025919109582901, 0.0...",0.715488,0.922509,0.700373
1,The Audit Senior Manager/Audit Manager manages...,The Audit Manager oversees a range of audit pr...,The Tax Associate is responsible for preparing...,"[0.057781320065259933, 0.07172407954931259, -0...","[0.044836368411779404, 0.06545188277959824, -0...","[0.0627947673201561, 0.07432186603546143, 0.01...",0.622581,0.901748,0.559541
2,The Audit Partner/Audit Director is a transfor...,The Audit Partner/Audit Director serves as a v...,The Tax Partner is a senior professional respo...,"[0.03437991812825203, 0.07796177268028259, -0....","[0.027283621951937675, 0.07725761830806732, -0...","[0.03415120393037796, 0.08369773626327515, -0....",0.743910,0.955510,0.709893
3,The Audit Senior is expected to team lead vari...,The Audit Senior leads audit engagements of va...,The Tax Associate is responsible for preparing...,"[0.02032621204853058, 0.06756345182657242, -0....","[0.006458158604800701, 0.07473322004079819, -0...","[0.06271377950906754, 0.07089059799909592, 0.0...",0.630662,0.919201,0.597181
4,The Business Valuation Associate/Business Valu...,The Business Valuation Executive is primarily ...,1. Easy Negative - Different Function: \nThe ...,"[0.032666150480508804, 0.05040699988603592, -0...","[0.014305354095995426, 0.04852457717061043, -0...","[0.030105773359537125, 0.02217748947441578, 0....",0.664242,0.899370,0.708093
...,...,...,...,...,...,...,...,...,...
3765,The WSH Manager is responsible for reviewing W...,The WSH Manager oversees the development and c...,The WSH Coordinator supports the implementatio...,"[-0.013950522057712078, 0.08462847769260406, -...","[-0.003074741456657648, 0.06595790386199951, -...","[0.005600765813142061, 0.05437818169593811, -0...",0.866656,0.908244,0.848154
3766,The WSH Officer is responsible for developing ...,The WSH Officer oversees the establishment and...,The Safety Training Coordinator is tasked with...,"[-0.0010848849778994918, 0.05022452771663666, ...","[0.009872515685856342, 0.048400189727544785, -...","[0.039964500814676285, 0.022148828953504562, -...",0.687135,0.912035,0.676861
3767,The Workplace Safety and Health (WSH) Supervis...,The Workplace Safety and Health (WSH) Supervis...,The Workplace Safety and Health (WSH) Coordina...,"[0.020672127604484558, 0.08999120444059372, -0...","[0.011292374692857265, 0.09653736650943756, -0...","[0.05170255899429321, 0.06624729186296463, -0....",0.850245,0.962092,0.847947
3768,The Lead Workplace Safety and Health (WSH) Aud...,The Lead Workplace Safety and Health (WSH) Aud...,The Lead Environmental Compliance Officer mana...,"[-0.003643264528363943, 0.04361746087670326, -...","[-0.0033322880044579506, 0.06397241353988647, ...","[0.037811484187841415, 0.061270490288734436, -...",0.640458,0.954095,0.632564


# Apply Quality Filtering Criteria
### Filtering Rules:
1. Anchor-Positive similarity > 0.6: Ensures positive examples are semantically related to anchors
2. Anchor-Negative similarity 0.2-0.6: Creates "hard negatives" that are somewhat related but not too similar
3. Positive-Negative similarity < 0.7: Ensures positive and negative examples are sufficiently distinct

In [ ]:
def filter_with_hard_negatives(example):
    anchor_pos = example["similarity-anchor-positive"]
    anchor_neg = example["similarity-anchor-negative"] 
    pos_neg = example["similarity-positive-negative"]
    
    return (
        anchor_pos > 0.6 and  # Good positive
        0.2 < anchor_neg < 0.6 and  # Hard negative range
        pos_neg < 0.7  # Positive and negative are distinct
    )

    # return (
    #     anchor_pos > 0.78 and  # Good positive
    #     0.2 < anchor_neg < 0.78 and  # Hard negative range
    #     pos_neg < 0.7  # Positive and negative are distinct (ask dickson why need this)
    # )


cleaned_dataset = dataset.filter(filter_with_hard_negatives)

Filter:   0%|          | 0/5655 [00:00<?, ? examples/s]

# Clean Filtered Dataset
### Remove the embedding vectors and similarity scores to keep only the essential text data. This reduces storage requirements while preserving the high-quality triplets identified by our filtering process.

In [9]:
cleaned_dataset = cleaned_dataset.select_columns(['anchor', 'positive', 'negative'])
cleaned_dataset

Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 1407
})

# Split into Training and Validation Sets

In [10]:
train_size = int(0.8 * len(cleaned_dataset))
valid_size = len(cleaned_dataset) - train_size

train_dataset = cleaned_dataset.select(range(train_size))
valid_dataset = cleaned_dataset.select(range(train_size, train_size + valid_size))

# Verify Dataset Splits

In [11]:
train_dataset

Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 1125
})

In [12]:
valid_dataset

Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 282
})

In [13]:
from datasets import DatasetDict

ds = DatasetDict({
    "train": train_dataset,
    "valid": valid_dataset
})

ds

DatasetDict({
    train: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 1125
    })
    valid: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 282
    })
})

# Create Dataset Dictionary

In [14]:
ds.push_to_hub("frankwong2001/ssf-train-valid-full-synthetic-batch10-cleaned")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :  45%|####5     |  527kB / 1.16MB            

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        : 100%|##########|  240kB /  240kB            

CommitInfo(commit_url='https://huggingface.co/datasets/frankwong2001/ssf-train-valid-full-synthetic-batch10-cleaned/commit/7ed5d68d8ca15ee4c388940a248a69b5d8f345b9', commit_message='Upload dataset', commit_description='', oid='7ed5d68d8ca15ee4c388940a248a69b5d8f345b9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/frankwong2001/ssf-train-valid-full-synthetic-batch10-cleaned', endpoint='https://huggingface.co', repo_type='dataset', repo_id='frankwong2001/ssf-train-valid-full-synthetic-batch10-cleaned'), pr_revision=None, pr_num=None)